# ChEMBL Adapter Example
Demonstrates searching and retrieving molecule details using the ChEMBLAdapter.

In [54]:
from aid_pais_knowledgegraph.knowledge_lookup.adapters.chembl_adapter import ChEMBLAdapter
from aid_pais_knowledgegraph.knowledge_lookup.models import LookupConfig

import logging

# Suppress verbose ChEMBL client logging
logging.getLogger('chembl_webresource_client.url_query').setLevel(logging.WARNING)
logging.getLogger('chembl_webresource_client').setLevel(logging.WARNING)

config = LookupConfig()
adapter = ChEMBLAdapter(config)

adapter.check_api_status()

{'available': True,
 'status_code': None,
 'error': None,
 'endpoints_tested': ['status(not available)', 'molecule', 'activity'],
 'available_endpoints': ['activity',
  'assay',
  'atc_class',
  'binding_site',
  'biotherapeutic',
  'cell_line',
  'chembl_id_lookup',
  'compound_record',
  'compound_structural_alert',
  'document',
  'document_similarity',
  'drug',
  'drug_indication',
  'drug_warning',
  'go_slim',
  'image',
  'mechanism',
  'metabolism',
  'molecule',
  'molecule_form',
  'organism',
  'similarity',
  'source',
  'substructure',
  'target',
  'target_component',
  'target_relation',
  'tissue',
  'xref_source']}

In [55]:
# Utility function to demo a ChEMBL endpoint

def demo_chembl_endpoint(adapter, endpoint):
    client = getattr(adapter.chembl_client, endpoint, None)
    if client is not None:
        try:
            if hasattr(client, 'all'):
                data = client.all()[:1]
            elif hasattr(client, 'filter'):
                data = client.filter()[0:1]
            else:
                data = str(client)
            print(f'Endpoint: {endpoint}\nResult: {data}\n---')
        except Exception as e:
            print(f'Endpoint: {endpoint}\nError: {e}\n---')
    else:
        print(f'Endpoint: {endpoint} not available in client\n---')

In [56]:

# Test with increased retries
print("Testing activity endpoint with retry logic...")
try:
    # Add filters to speed up query and get more relevant results
    filters = {
        'molecule_chembl_id': 'CHEMBL25',  # Specific molecule (aspirin)
        'standard_type__exact': 'IC50',   # Only IC50 measurements
        'pchembl_value__isnull': False,   # Must have potency values
        'assay_type': 'B',                # Binding assays only
        'confidence_score__gte': 8,       # High confidence assays
        'data_validity_comment__isnull': True,  # Exclude invalid data
        'standard_relation__exact': '='   # Exact measurements only
    }
    activities = adapter.lookup_activity(filters=filters)
    print(f"Success! Retrieved {len(activities)} activities")
    if activities:
        print(f"Sample activity: {activities[0]}")
except Exception as e:
    print(f"Still failed after retries: {e}")

print()


Testing activity endpoint with retry logic...
Success! Retrieved 68 activities
Sample activity: {'action_type': None, 'activity_comment': None, 'activity_id': 91852, 'activity_properties': [], 'assay_chembl_id': 'CHEMBL760085', 'assay_description': 'Inhibitory concentration in DMSO with purified human Prostaglandin G/H synthase 2 (COX-2)', 'assay_type': 'B', 'assay_variant_accession': None, 'assay_variant_mutation': None, 'bao_endpoint': 'BAO_0000190', 'bao_format': 'BAO_0000357', 'bao_label': 'single protein format', 'canonical_smiles': 'CC(=O)Oc1ccccc1C(=O)O', 'data_validity_comment': None, 'data_validity_description': None, 'document_chembl_id': 'CHEMBL1131194', 'document_journal': 'J Med Chem', 'document_year': 1998, 'ligand_efficiency': {'bei': '23.34', 'le': '0.44', 'lle': '2.89', 'sei': '6.61'}, 'molecule_chembl_id': 'CHEMBL25', 'molecule_pref_name': 'ASPIRIN', 'parent_molecule_chembl_id': 'CHEMBL25', 'pchembl_value': '4.20', 'potential_duplicate': 0, 'qudt_units': 'http://www.o

In [57]:
# Demo: activity_supplementary_data_by_activity endpoint

demo_chembl_endpoint(adapter, 'activity_supplementary_data_by_activity')

Endpoint: activity_supplementary_data_by_activity
Result: [{'action_type': None, 'activity_comment': 'See Activity_Supp For Individual Animal Data', 'activity_id': 17126237, 'activity_properties': [{'comments': None, 'relation': None, 'result_flag': 0, 'standard_relation': None, 'standard_text_value': 'RBC (Erythrocytes)', 'standard_type': 'ACTIVITY_TEST', 'standard_units': None, 'standard_value': None, 'text_value': 'RBC (Erythrocytes)', 'type': 'ACTIVITY_TEST', 'units': None, 'value': None}, {'comments': None, 'relation': None, 'result_flag': 0, 'standard_relation': None, 'standard_text_value': 'Hematology', 'standard_type': 'DATASET', 'standard_units': None, 'standard_value': None, 'text_value': 'Hematology', 'type': 'DATASET', 'units': None, 'value': None}, {'comments': None, 'relation': '=', 'result_flag': 0, 'standard_relation': '=', 'standard_text_value': None, 'standard_type': 'DOSE', 'standard_units': 'mg.kg-1', 'standard_value': '0.0', 'text_value': None, 'type': 'DOSE', 'uni

In [58]:
# Demo: assay endpoint

demo_chembl_endpoint(adapter, 'assay')

Endpoint: assay
Result: [{'aidx': 'CLD0', 'assay_category': None, 'assay_cell_type': None, 'assay_chembl_id': 'CHEMBL615117', 'assay_classifications': [], 'assay_group': None, 'assay_organism': None, 'assay_parameters': [], 'assay_strain': None, 'assay_subcellular_fraction': None, 'assay_tax_id': None, 'assay_test_type': None, 'assay_tissue': None, 'assay_type': 'B', 'assay_type_description': 'Binding', 'bao_format': 'BAO_0000019', 'bao_label': 'assay format', 'cell_chembl_id': None, 'confidence_description': 'Homologous single protein target assigned', 'confidence_score': 8, 'description': 'The compound was tested for the in vitro inhibition of platelet 12-lipoxygenase at a concentration of 30 uM', 'document_chembl_id': 'CHEMBL1125582', 'relationship_description': 'Homologous protein target assigned', 'relationship_type': 'H', 'src_assay_id': None, 'src_id': 1, 'target_chembl_id': 'CHEMBL2741', 'tissue_chembl_id': None, 'variant_sequence': None}]
---


In [59]:
# Demo: assay_class endpoint

demo_chembl_endpoint(adapter, 'assay_class')

Endpoint: assay_class
Result: [{'assay_class_id': 1, 'bao_id': None, 'class_type': 'In vivo efficacy', 'l1': 'ALIMENTARY TRACT AND METABOLISM', 'l2': 'Anti-Obesity Activity', 'l3': 'Computer-Assisted Measurement of Food Consumption in Rats Anorectic Activity', 'source': 'Vogel_2008'}]
---


In [60]:
# Demo: atc_class endpoint

demo_chembl_endpoint(adapter, 'atc_class')

Endpoint: atc_class
Result: [{'level1': 'A', 'level1_description': 'ALIMENTARY TRACT AND METABOLISM', 'level2': 'A01', 'level2_description': 'STOMATOLOGICAL PREPARATIONS', 'level3': 'A01A', 'level3_description': 'STOMATOLOGICAL PREPARATIONS', 'level4': 'A01AA', 'level4_description': 'Caries prophylactic agents', 'level5': 'A01AA01', 'who_name': 'sodium fluoride'}]
---


In [61]:
# Demo: binding_site endpoint

demo_chembl_endpoint(adapter, 'binding_site')

Endpoint: binding_site
Result: [{'site_components': [{'component_id': 4909, 'domain': {'domain_description': None, 'domain_id': 2781, 'domain_name': 'UDPGT', 'domain_type': 'Pfam-A', 'source_domain_id': 'PF00201'}, 'sitecomp_id': 4}], 'site_id': 2, 'site_name': 'UDP-glucuronosyltransferase 1-10, UDPGT domain'}]
---


In [62]:
# Demo: biotherapeutic endpoint

demo_chembl_endpoint(adapter, 'biotherapeutic')

Endpoint: biotherapeutic
Result: [{'biocomponents': [], 'description': None, 'helm_notation': 'PEPTIDE1{A.L.Y.A.S.K.L.S.[am]}$$$$', 'molecule_chembl_id': 'CHEMBL448105'}]
---


In [63]:
# Demo: cell_line endpoint

demo_chembl_endpoint(adapter, 'cell_line')

Endpoint: cell_line
Result: [{'cell_chembl_id': 'CHEMBL3307241', 'cell_description': 'DC3F', 'cell_id': 1, 'cell_name': 'DC3F', 'cell_source_organism': 'Cricetulus griseus', 'cell_source_tax_id': 10029, 'cell_source_tissue': 'Lung', 'cellosaurus_id': 'CVCL_4704', 'cl_lincs_id': None, 'clo_id': None, 'efo_id': None}]
---


In [64]:
# Demo: chembl_id_lookup endpoint

demo_chembl_endpoint(adapter, 'chembl_id_lookup')

Endpoint: chembl_id_lookup
Result: [{'chembl_id': 'CHEMBL1', 'entity_type': 'COMPOUND', 'last_active': 28, 'resource_url': '/chembl/api/data/molecule/CHEMBL1', 'status': 'ACTIVE'}]
---


In [65]:
# Demo: chembl_release endpoint

demo_chembl_endpoint(adapter, 'chembl_release')

Endpoint: chembl_release
Result: [{'chembl_release': 'CHEMBL_1', 'creation_date': '2009-09-03'}]
---


In [66]:
# Demo: compound_record endpoint

demo_chembl_endpoint(adapter, 'compound_record')

Endpoint: compound_record
Result: [{'compound_key': 'X', 'compound_name': 'Bis(3-[14-Benzyl-11-(1H-indol-3-ylmethyl)-2-isobutyl-13-methyl-5-(2-methylsulfanyl-ethyl)-3,6,9,12,15,20-hexaoxo-1,4,7,10,13,16-hexaaza-bicyclo[15.2.1]icos-18-en-8-yl]-propionamide)', 'document_chembl_id': 'CHEMBL1126670', 'molecule_chembl_id': 'CHEMBL3349687', 'record_id': 1, 'src_id': 1}]
---


In [67]:
# Demo: compound_structural_alert endpoint

demo_chembl_endpoint(adapter, 'compound_structural_alert')

Endpoint: compound_structural_alert
Result: [{'alert': {'alert_id': 1, 'alert_name': 'R1 Reactive alkyl halides', 'alert_set': {'priority': 8, 'set_name': 'Glaxo'}, 'smarts': '[Br,Cl,I][CX4;CH,CH2]'}, 'cpd_str_alert_id': 83590421, 'molecule_chembl_id': 'CHEMBL266081'}]
---


In [68]:
# Demo: description endpoint

demo_chembl_endpoint(adapter, 'description')

Endpoint: description
Result: {'expected_status': [200], 'version': 'data', 'name': 'ChEMBL web services API live documentation', 'methods': {'GET_activity_api_dispatch_list': {'method': 'GET', 'resource_name': 'activity', 'schema': '/chembl/api/data/activity/schema', 'collection_name': 'activities', 'default_format': 'application/xml', 'description': 'Retrieve activity object list.', 'path': '/chembl/api/data/activity/', 'required_params': [], 'formats': ['xml', 'json', 'jsonp', 'yaml']}, 'GET_activity_api_dispatch_detail': {'method': 'GET', 'resource_name': 'activity', 'schema': '/chembl/api/data/activity/schema', 'collection_name': 'activities', 'default_format': 'application/xml', 'description': 'Retrieve single activity object details by ID.', 'path': '/chembl/api/data/activity/:ID', 'required_params': ['ID'], 'formats': ['xml', 'json', 'jsonp', 'yaml']}, 'GET_activity_api_get_multiple': {'method': 'GET', 'resource_name': 'activity', 'schema': '/chembl/api/data/activity/schema', '

In [69]:
# Demo: document endpoint

demo_chembl_endpoint(adapter, 'document')

Endpoint: document
Result: [{'abstract': '', 'authors': None, 'chembl_release': {'chembl_release': 'CHEMBL_7', 'creation_date': '2010-09-29'}, 'contact': None, 'doc_type': 'DATASET', 'document_chembl_id': 'CHEMBL1158643', 'doi': None, 'doi_chembl': None, 'first_page': None, 'issue': None, 'journal': None, 'journal_full_title': None, 'last_page': None, 'patent_id': None, 'pubmed_id': None, 'src_id': 0, 'title': 'Unpublished dataset', 'volume': None, 'year': None}]
---


In [70]:
# Demo: document_similarity endpoint

demo_chembl_endpoint(adapter, 'document_similarity')

Endpoint: document_similarity
Result: [{'document_1_chembl_id': 'CHEMBL1148466', 'document_2_chembl_id': 'CHEMBL1144014', 'mol_tani': 0.0, 'tid_tani': 1.0}]
---


In [71]:
# Demo: drug endpoint

demo_chembl_endpoint(adapter, 'drug')

Endpoint: drug
Result: [{'applicants': ['Alembic Pharmaceuticals Ltd', 'Novitium Pharma Llc', 'Lannett Co Inc', 'Msn Laboratories Private Ltd', 'Granules Pharmaceuticals Inc', 'Mylan Pharmaceuticals Inc', 'Appco Pharma Llc', 'Ani Pharmaceuticals Inc', 'Watson Laboratories Inc', 'Teva Pharmaceuticals Usa'], 'atc_classification': [{'code': 'C02CA01', 'description': 'CARDIOVASCULAR SYSTEM: ANTIHYPERTENSIVES: ANTIADRENERGIC AGENTS, PERIPHERALLY ACTING: Alpha-adrenoreceptor antagonists'}], 'availability_type': 1, 'biotherapeutic': None, 'black_box': 0, 'black_box_warning': '0', 'chirality': 2, 'drug_type': 1, 'first_approval': 1976, 'first_in_class': 0, 'helm_notation': None, 'max_phase': '4.0', 'molecule_chembl_id': 'CHEMBL2', 'molecule_properties': {'alogp': '1.78', 'aromatic_rings': 3, 'full_molformula': 'C19H21N5O4', 'full_mwt': '383.41', 'hba': 8, 'hbd': 1, 'heavy_atoms': 28, 'mw_freebase': '383.41', 'np_likeness_score': '-1.29', 'num_ro5_violations': 0, 'psa': '106.95', 'qed_weighted'

In [72]:
# Demo: drug_indication endpoint

demo_chembl_endpoint(adapter, 'drug_indication')

Endpoint: drug_indication
Result: [{'drugind_id': 22606, 'efo_id': 'EFO:0000404', 'efo_term': 'diffuse scleroderma', 'indication_refs': [{'ref_id': 'NCT00442611,NCT02161406', 'ref_type': 'ClinicalTrials', 'ref_url': 'https://clinicaltrials.gov/search?term=NCT00442611%20NCT02161406'}], 'max_phase_for_ind': '2.0', 'mesh_heading': 'Scleroderma, Diffuse', 'mesh_id': 'D045743', 'molecule_chembl_id': 'CHEMBL1201823', 'parent_molecule_chembl_id': 'CHEMBL1201823'}]
---


In [73]:
# Demo: drug_warning endpoint

demo_chembl_endpoint(adapter, 'drug_warning')

Endpoint: drug_warning
Result: [{'efo_id': None, 'efo_id_for_warning_class': 'EFO:0011052', 'efo_term': None, 'molecule_chembl_id': 'CHEMBL4303288', 'parent_molecule_chembl_id': 'CHEMBL1380', 'warning_class': 'hepatotoxicity', 'warning_country': 'United States', 'warning_description': None, 'warning_id': 1, 'warning_refs': [{'ref_id': 'de109a2b-e36c-40d0-85fc-a67a9e7f1ae8', 'ref_type': 'DailyMed', 'ref_url': 'https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid=de109a2b-e36c-40d0-85fc-a67a9e7f1ae8'}, {'ref_id': '73ce0bc5-43e2-c57f-0a6a-bdaf9fbaa3c2', 'ref_type': 'DailyMed', 'ref_url': 'https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid=73ce0bc5-43e2-c57f-0a6a-bdaf9fbaa3c2'}, {'ref_id': 'e5cbf204-c10d-444b-aba0-180a30645d55', 'ref_type': 'DailyMed', 'ref_url': 'https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid=e5cbf204-c10d-444b-aba0-180a30645d55'}, {'ref_id': 'ca73b519-015a-436d-aa3c-af53492825a1', 'ref_type': 'DailyMed', 'ref_url': 'https://dailymed.nlm.nih.gov/dailym

In [74]:
# Demo: go_slim endpoint

demo_chembl_endpoint(adapter, 'go_slim')

Endpoint: go_slim
Result: [{'aspect': 'P', 'class_level': 1, 'go_id': 'GO:0000003', 'parent_go_id': 'GO:0008150', 'path': 'biological_process  reproduction', 'pref_name': 'reproduction'}]
---


In [75]:
# Demo: image endpoint

demo_chembl_endpoint(adapter, 'image')

Endpoint: image
Error: 'NoneType' object is not subscriptable
---


In [76]:
# Demo: mechanism endpoint

demo_chembl_endpoint(adapter, 'mechanism')

Endpoint: mechanism
Result: [{'action_type': 'INHIBITOR', 'binding_site_comment': None, 'direct_interaction': 1, 'disease_efficacy': 1, 'max_phase': 4, 'mec_id': 13, 'mechanism_comment': None, 'mechanism_of_action': 'Carbonic anhydrase VII inhibitor', 'mechanism_refs': [{'ref_id': 'setid=8e162b6d-8fa6-45f6-80d8-5132d94c1207', 'ref_type': 'DailyMed', 'ref_url': 'http://dailymed.nlm.nih.gov/dailymed/lookup.cfm?setid=8e162b6d-8fa6-45f6-80d8-5132d94c1207'}, {'ref_id': '18336310', 'ref_type': 'PubMed', 'ref_url': 'http://europepmc.org/abstract/MED/18336310'}], 'molecular_mechanism': 1, 'molecule_chembl_id': 'CHEMBL19', 'parent_molecule_chembl_id': 'CHEMBL19', 'record_id': 1343810, 'selectivity_comment': None, 'site_id': None, 'target_chembl_id': 'CHEMBL2326', 'variant_sequence': None}]
---


In [77]:
# Demo: metabolism endpoint

demo_chembl_endpoint(adapter, 'metabolism')

Endpoint: metabolism
Result: [{'drug_chembl_id': 'CHEMBL417', 'enzyme_name': None, 'met_comment': None, 'met_conversion': None, 'met_id': 119, 'metabolism_refs': [{'ref_id': 'http://www.accessdata.fda.gov/drugsatfda_docs/nda/99/50-778_Ellence_biopharmr.pdf', 'ref_type': 'OTHER', 'ref_url': 'http://www.accessdata.fda.gov/drugsatfda_docs/nda/99/50-778_Ellence_biopharmr.pdf'}], 'metabolite_chembl_id': 'CHEMBL3508152', 'metabolite_name': 'Doxorubicinol aglycone', 'organism': 'Homo sapiens', 'pathway_id': 1, 'pathway_key': 'Fig. 2, p.19', 'substrate_chembl_id': 'CHEMBL3508141', 'substrate_name': "Epidoxorubicinol, 4'-epiadriamycinol", 'target_chembl_id': None, 'tax_id': 9606}]
---


In [78]:
# Demo: molecule endpoint

demo_chembl_endpoint(adapter, 'molecule')

Endpoint: molecule
Result: [{'atc_classifications': [], 'availability_type': -1, 'biotherapeutic': None, 'black_box_warning': 0, 'chemical_probe': 0, 'chirality': -1, 'cross_references': [], 'dosed_ingredient': False, 'first_approval': None, 'first_in_class': -1, 'helm_notation': None, 'inorganic_flag': -1, 'max_phase': None, 'molecule_chembl_id': 'CHEMBL6329', 'molecule_hierarchy': {'active_chembl_id': 'CHEMBL6329', 'molecule_chembl_id': 'CHEMBL6329', 'parent_chembl_id': 'CHEMBL6329'}, 'molecule_properties': {'alogp': '2.11', 'aromatic_rings': 3, 'full_molformula': 'C17H12ClN3O3', 'full_mwt': '341.75', 'hba': 5, 'hbd': 1, 'heavy_atoms': 24, 'mw_freebase': '341.75', 'np_likeness_score': '-1.56', 'num_ro5_violations': 0, 'psa': '84.82', 'qed_weighted': '0.74', 'ro3_pass': 'N', 'rtb': 3}, 'molecule_structures': {'canonical_smiles': 'Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccccc1Cl', 'molfile': '\n     RDKit          2D\n\n 24 26  0  0  0  0  0  0  0  0999 V2000\n    5.2792   -2.0500    0.000

In [79]:
# Demo: molecule_form endpoint

demo_chembl_endpoint(adapter, 'molecule_form')

Endpoint: molecule_form
Result: [{'is_parent': True, 'molecule_chembl_id': 'CHEMBL6329', 'parent_chembl_id': 'CHEMBL6329'}]
---


In [80]:
# Demo: official endpoint

demo_chembl_endpoint(adapter, 'official')

Endpoint: official
Result: False
---


In [81]:
# Demo: organism endpoint

demo_chembl_endpoint(adapter, 'organism')

Endpoint: organism
Result: [{'l1': 'Eukaryotes', 'l2': 'Mammalia', 'l3': 'Rodentia', 'oc_id': 1, 'tax_id': 10030}]
---


In [82]:
# Demo: protein_classification endpoint

demo_chembl_endpoint(adapter, 'protein_classification')

Endpoint: protein_classification
Result: [{'class_level': 0, 'definition': 'Root of the ChEMBL protein family classification', 'parent_id': None, 'pref_name': 'Protein class', 'protein_class_desc': 'protein class', 'protein_class_id': 0, 'replaced_by': None, 'short_name': 'Protein class', 'sort_order': None}]
---


In [83]:
# Demo: similarity endpoint

demo_chembl_endpoint(adapter, 'similarity')

Endpoint: similarity
Error: Error for url https://www.ebi.ac.uk/chembl/api/data/similarity.json, server response: {"error_message": "Similarity parameter is required."}
---


In [84]:
# Demo: source endpoint

demo_chembl_endpoint(adapter, 'source')

Endpoint: source
Result: [{'src_comment': None, 'src_description': 'Undefined', 'src_id': 0, 'src_short_name': 'UNDEFINED', 'src_url': None}]
---


In [85]:
# Demo: substructure endpoint

demo_chembl_endpoint(adapter, 'substructure')

Endpoint: substructure
Error: Error for url https://www.ebi.ac.uk/chembl/api/data/substructure.json, server response: {"error_message": "Structure or identifier required."}
---


In [86]:
# Demo: target endpoint

demo_chembl_endpoint(adapter, 'target')

Endpoint: target
Result: [{'cross_references': [], 'organism': 'Homo sapiens', 'pref_name': 'Maltase-glucoamylase', 'species_group_flag': False, 'target_chembl_id': 'CHEMBL2074', 'target_components': [{'accession': 'O43451', 'component_description': 'Maltase-glucoamylase', 'component_id': 434, 'component_type': 'PROTEIN', 'relationship': 'SINGLE PROTEIN', 'target_component_synonyms': [{'component_synonym': '3.2.1.20', 'syn_type': 'EC_NUMBER'}, {'component_synonym': 'Alpha-1,4-glucosidase', 'syn_type': 'UNIPROT'}, {'component_synonym': 'Maltase-glucoamylase', 'syn_type': 'UNIPROT'}, {'component_synonym': 'MGA', 'syn_type': 'GENE_SYMBOL_OTHER'}, {'component_synonym': 'MGAM', 'syn_type': 'GENE_SYMBOL'}, {'component_synonym': 'MGAML', 'syn_type': 'GENE_SYMBOL_OTHER'}, {'component_synonym': 'Synonyms=MGA', 'syn_type': 'GENE_SYMBOL_OTHER'}], 'target_component_xrefs': [{'xref_id': 'O43451', 'xref_name': None, 'xref_src_db': 'AlphaFoldDB'}, {'xref_id': 'O43451', 'xref_name': None, 'xref_src_db

In [87]:
# Demo: target_component endpoint

demo_chembl_endpoint(adapter, 'target_component')

Endpoint: target_component
Result: [{'accession': 'O09028', 'component_id': 1, 'component_type': 'PROTEIN', 'description': 'Gamma-aminobutyric acid receptor subunit pi', 'go_slims': [{'go_id': 'GO:0003674'}, {'go_id': 'GO:0005575'}, {'go_id': 'GO:0005886'}, {'go_id': 'GO:0006810'}, {'go_id': 'GO:0016020'}, {'go_id': 'GO:0022857'}, {'go_id': 'GO:0055085'}], 'organism': 'Rattus norvegicus', 'protein_classifications': [{'protein_classification_id': 1173}], 'sequence': 'MSYSLYLAFVCLNLLAQRMCIQGNQFNVEVSRSDKLSLPGFENLTAGYNKFLRPNFGGDPVRIALTLDIASISSISESNMDYTATIYLRQRWTDPRLVFEGNKSFTLDARLVEFLWVPDTYIVESKKSFLHEVTVGNRLIRLFSNGTVLYALRITTTVTCNMDLSKYPMDTQTCKLQLESWGYDGNDVEFSWLRGNDSVRGLENLRLAQYTIQQYFTLVTVSQQETGNYTRLVLQFELRRNVLYFILETYVPSTFLVVLSWVSFWISLESVPARTCIGVTTVLSMTTLMIGSRTSLPNTNCFIKAIDVYLGICFSFVFGALLEYAVAHYSSLQQMAVKDRGPAKDSEEVNITNIINSSISSFKRKISFASIEISGDNVNYSDLTMKASDKFKFVFREKIGRIIDYFTIQNPSNVDRYSKLLFPLIFMLANVFYWAYYMYF', 'target_component_synonyms': [{'component_synonym': 'GABA(A) receptor subunit pi', 'sy

In [88]:
# Demo: target_relation endpoint

demo_chembl_endpoint(adapter, 'target_relation')

Endpoint: target_relation
Result: [{'related_target_chembl_id': 'CHEMBL2096619', 'relationship': 'SUBSET OF', 'target_chembl_id': 'CHEMBL2251'}]
---


In [89]:
# Demo: tissue endpoint

demo_chembl_endpoint(adapter, 'tissue')

Endpoint: tissue
Result: [{'bto_id': 'BTO:0001421', 'caloha_id': 'TS-0134', 'efo_id': 'EFO:0000979', 'pref_name': 'Uterine cervix', 'tissue_chembl_id': 'CHEMBL3988026', 'uberon_id': 'UBERON:0000002'}]
---


In [90]:
# Demo: xref_source endpoint

demo_chembl_endpoint(adapter, 'xref_source')

Endpoint: xref_source
Result: [{'xref_id_url': 'https://alphafold.ebi.ac.uk/entry/$$', 'xref_src_db': 'AlphaFoldDB', 'xref_src_description': 'Alpha Fold Protein Structure Database', 'xref_src_url': 'https://alphafold.ebi.ac.uk/'}]
---


## ChEMBLAdapter Unified Concept Search & Details Demo
Demonstrates the new search_concepts and get_concept_details methods, which search across molecule, drug, and target endpoints.

### About the `filters` Keyword in ChEMBLAdapter

The `filters` keyword is used to specify search criteria for ChEMBL endpoints. It is a dictionary where keys are field names (optionally with lookup modifiers) and values are the terms to match.

For example:
- `{"pref_name__icontains": "aspirin"}` will search for records where the preferred name contains "aspirin" (case-insensitive).
- You can use other lookup modifiers like `exact`, `startswith`, etc., e.g. `{"molecule_chembl_id__exact": "CHEMBL25"}`.

This allows flexible and powerful querying of ChEMBL data. See the [ChEMBL API documentation](https://chembl.gitbook.io/chembl-interface-documentation/web-services) for more details on available fields and lookup types.

In [91]:
# Demo: search_concepts (searches molecule, drug, target endpoints)
# Simple await-based demo for Jupyter
async def demo_search_concepts(adapter, query, limit=3):
    try:
        concepts = await adapter.search_concepts(query, limit=limit)
        for c in concepts:
            print(f"Type: {c.concept_type}, Label: {c.primary_label}, ID: {c.primary_id}")
            print(f"Categories: {c.categories}")
            print(f"Definitions: {c.definitions}")
            print(f"Identifiers: {[i.identifier for i in c.identifiers]}")
            print(c)
    except Exception as e:
        print(f"Error: {e}")

# Example search for 'aspirin'
await demo_search_concepts(adapter, "Aripiprazole", limit=3)

INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.ols_adapter:OLS search for 'small molecule' returned 5 concepts
INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.chembl_adapter:Mapped category 'Small molecule' to OLS label 'Small Molecule'
INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.chembl_adapter:Mapped category 'Small molecule' to OLS label 'Small Molecule'
INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.ols_adapter:OLS search for 'therapeutic' returned 5 concepts
INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.chembl_adapter:Mapped category 'Therapeutic' to OLS label 'Therapeutic'
INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.ols_adapter:OLS search for 'therapeutic' returned 5 concepts
INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.chembl_adapter:Mapped category 'Therapeutic' to OLS label 'Therapeutic'
INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.ols_adapter:OLS search for 'natural product' returned 5 concepts
INFO:aid_

Type: ConceptType.CHEMICAL, Label: ARIPIPRAZOLE, ID: CHEMBL1112
Categories: ['Small Molecule', 'Therapeutic', 'Natural Product', 'Oral', 'Parenteral']
Definitions: ['Type: Small molecule', 'Properties: MW: 448.39, Formula: C23H27Cl2N3O2, LogP: 4.86, PSA: 44.81, HBD: 1, HBA: 4, Aromatic rings: 2, RO3 compliant: N, QED: 0.61', 'Structures: Has SMILES, Has InChI, Has InChI Key', 'Natural product']
Identifiers: ['CHEMBL1112']
UnifiedConcept(primary_id='CHEMBL1112', primary_label='ARIPIPRAZOLE', concept_type=<ConceptType.CHEMICAL: 'chemical'>, identifiers=[ConceptIdentifier(source=<KnowledgeSource.CHEMBL: 'chembl'>, identifier='CHEMBL1112', label='ARIPIPRAZOLE', url='https://www.ebi.ac.uk/chembl/compound_report_card/CHEMBL1112/')], mappings=[], labels={}, synonyms=[], definitions=['Type: Small molecule', 'Properties: MW: 448.39, Formula: C23H27Cl2N3O2, LogP: 4.86, PSA: 44.81, HBD: 1, HBA: 4, Aromatic rings: 2, RO3 compliant: N, QED: 0.61', 'Structures: Has SMILES, Has InChI, Has InChI Key',

In [92]:
# Demo: get_concept_details (fetches details for a specific ChEMBL ID)


async def demo_get_concept_details(adapter, concept_id):
    concept = await adapter.get_concept_details(concept_id)
    if concept:
        print(f"Type: {concept.concept_type}, Label: {concept.primary_label}, ID: {concept.primary_id}")
        print(f"Categories: {concept.categories}")
        print(f"Definitions: {concept.definitions}")
        print(f"Identifiers: {[i.identifier for i in concept.identifiers]}")
        print(f"Full Concept: {concept}")
    else:
        print(f"No concept found for ID: {concept_id}")

# Example: details for CHEMBL25 (aspirin)
await demo_get_concept_details(adapter, "CHEMBL25")

INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.ols_adapter:OLS search for 'small molecule' returned 5 concepts
INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.chembl_adapter:Mapped category 'Small molecule' to OLS label 'Small Molecule'
INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.chembl_adapter:Mapped category 'Small molecule' to OLS label 'Small Molecule'
INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.ols_adapter:OLS search for 'therapeutic' returned 5 concepts
INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.chembl_adapter:Mapped category 'Therapeutic' to OLS label 'Therapeutic'
INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.ols_adapter:OLS search for 'therapeutic' returned 5 concepts
INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.chembl_adapter:Mapped category 'Therapeutic' to OLS label 'Therapeutic'
INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.ols_adapter:OLS search for 'natural product' returned 5 concepts
INFO:aid_

Type: ConceptType.CHEMICAL, Label: ASPIRIN, ID: CHEMBL25
Categories: ['Small Molecule', 'Therapeutic', 'Natural Product', 'Oral']
Definitions: ['Type: Small molecule', 'Properties: MW: 180.16, Formula: C9H8O4, LogP: 1.31, PSA: 63.60, HBD: 1, HBA: 3, Aromatic rings: 1, RO3 compliant: N, QED: 0.55', 'Structures: Has SMILES, Has InChI, Has InChI Key', 'Natural product']
Identifiers: ['CHEMBL25']
Full Concept: UnifiedConcept(primary_id='CHEMBL25', primary_label='ASPIRIN', concept_type=<ConceptType.CHEMICAL: 'chemical'>, identifiers=[ConceptIdentifier(source=<KnowledgeSource.CHEMBL: 'chembl'>, identifier='CHEMBL25', label='ASPIRIN', url='https://www.ebi.ac.uk/chembl/compound_report_card/CHEMBL25/')], mappings=[], labels={}, synonyms=[], definitions=['Type: Small molecule', 'Properties: MW: 180.16, Formula: C9H8O4, LogP: 1.31, PSA: 63.60, HBD: 1, HBA: 3, Aromatic rings: 1, RO3 compliant: N, QED: 0.55', 'Structures: Has SMILES, Has InChI, Has InChI Key', 'Natural product'], semantic_types=[],

## Exporting Activity Data to RDF

This example shows how to convert ChEMBL activity data to RDF triples using the `rdflib` library.

In [100]:
# Create RDF from Activity Data
from rdflib import Graph, URIRef, Literal, Namespace, RDF, RDFS
from rdflib.namespace import XSD

# Define AID-PAIS ontology namespace
AIDPAIS = Namespace("http://aid-pais.org/ontology#")
CHEMBL_ACTIVITY = Namespace("https://www.ebi.ac.uk/chembl/explore/activity/")
CHEMBL_ASSAY = Namespace("https://www.ebi.ac.uk/chembl/explore/assay/")
CHEMBL_COMPOUND = Namespace("https://www.ebi.ac.uk/chembl/explore/compound/")
CHEMBL_TARGET = Namespace("https://www.ebi.ac.uk/chembl/explore/target/")
CHEBI = Namespace("http://purl.obolibrary.org/obo/")
UNIT = Namespace("http://aid-pais.org/data/unit/")

def create_activity_rdf(activities):
    """
    Convert ChEMBL activity data to RDF triples using AID-PAIS ontology.
    """
    g = Graph()

    # Bind namespaces
    g.bind("aidpais", AIDPAIS)
    g.bind("chembl-activity", CHEMBL_ACTIVITY)
    g.bind("chembl-assay", CHEMBL_ASSAY)
    g.bind("chembl-compound", CHEMBL_COMPOUND)
    g.bind("chembl-target", CHEMBL_TARGET)
    g.bind("chebi", CHEBI)
    g.bind("unit", UNIT)

    for activity in activities[:1]:  # Limit to first 5 for demo
        # Create observation URI for the activity using official ChEMBL URL
        observation_uri = CHEMBL_ACTIVITY[f"CHEMBL{activity['activity_id']}"]

        # Activity as an Observation
        g.add((observation_uri, RDF.type, AIDPAIS.Observation))
        g.add((observation_uri, RDFS.label, Literal(f"ChEMBL Activity CHEMBL{activity['activity_id']}", datatype=XSD.string)))

        # Molecule as MolecularEntity using ChEBI molecular entity class
        if activity.get('molecule_chembl_id'):
            molecule_uri = CHEMBL_COMPOUND[activity['molecule_chembl_id']]
            g.add((molecule_uri, RDF.type, CHEBI.CHEBI_23367))  # ChEBI molecular entity
            g.add((observation_uri, AIDPAIS.hasParticipant, molecule_uri))
            if activity.get('molecule_pref_name'):
                g.add((molecule_uri, RDFS.label, Literal(activity['molecule_pref_name'], datatype=XSD.string)))

        # Target as Protein/MolecularEntity using official ChEMBL URL
        if activity.get('target_chembl_id'):
            target_uri = CHEMBL_TARGET[activity['target_chembl_id']]
            g.add((target_uri, RDF.type, AIDPAIS.Protein))  # Assuming targets are proteins
            g.add((observation_uri, AIDPAIS.hasParticipant, target_uri))
            if activity.get('target_pref_name'):
                g.add((target_uri, RDFS.label, Literal(activity['target_pref_name'], datatype=XSD.string)))

        # Assay relationship using official ChEMBL URL
        if activity.get('assay_chembl_id'):
            assay_uri = CHEMBL_ASSAY[activity['assay_chembl_id']]
            g.add((assay_uri, RDF.type, AIDPAIS.Assay))
            g.add((observation_uri, AIDPAIS.hasParticipant, assay_uri))

        # Activity measurements using AID-PAIS properties
        if activity.get('standard_type'):
            g.add((observation_uri, AIDPAIS.hasValue,
                  Literal(activity['standard_type'], datatype=XSD.string)))

        if activity.get('standard_value'):
            g.add((observation_uri, AIDPAIS.hasValue,
                  Literal(activity['standard_value'], datatype=XSD.float)))

        if activity.get('standard_units'):
            unit_uri = UNIT[activity['standard_units'].lower()]
            g.add((observation_uri, AIDPAIS.hasUnit, unit_uri))

        if activity.get('pchembl_value'):
            g.add((observation_uri, AIDPAIS.hasValue,
                  Literal(activity['pchembl_value'], datatype=XSD.float)))

        # Confidence score as evidence
        if activity.get('confidence_score'):
            evidence_uri = URIRef(f"http://aid-pais.org/data/evidence/chembl_confidence/{activity['activity_id']}")
            g.add((observation_uri, AIDPAIS.hasEvidence, evidence_uri))
            g.add((evidence_uri, AIDPAIS.hasValue, Literal(activity['confidence_score'], datatype=XSD.integer)))

    return g

# Create RDF from the activity data we retrieved
if 'activities' in locals() and activities:
    print(f"Creating RDF from {len(activities)} activities...")
    rdf_graph = create_activity_rdf(activities)

    # Serialize to Turtle format
    turtle_data = rdf_graph.serialize(format='turtle')
    print(turtle_data)
else:
    print("No activity data available. Run the activity query first.")

Creating RDF from 68 activities...
@prefix aidpais: <http://aid-pais.org/ontology#> .
@prefix chebi: <http://purl.obolibrary.org/obo/> .
@prefix chembl-activity: <https://www.ebi.ac.uk/chembl/explore/activity/> .
@prefix chembl-assay: <https://www.ebi.ac.uk/chembl/explore/assay/> .
@prefix chembl-compound: <https://www.ebi.ac.uk/chembl/explore/compound/> .
@prefix chembl-target: <https://www.ebi.ac.uk/chembl/explore/target/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix unit: <http://aid-pais.org/data/unit/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

chembl-activity:CHEMBL91852 a aidpais:Observation ;
    rdfs:label "ChEMBL Activity CHEMBL91852"^^xsd:string ;
    aidpais:hasParticipant chembl-assay:CHEMBL760085,
        chembl-compound:CHEMBL25,
        chembl-target:CHEMBL230 ;
    aidpais:hasUnit unit:nm ;
    aidpais:hasValue "4.2"^^xsd:float,
        "62500.0"^^xsd:float,
        "IC50"^^xsd:string .

chembl-assay:CHEMBL760085 a aidpais:Assay .

chemb